# SegmentAnyTree — Quick Start

This notebook runs tree instance segmentation on your point cloud files using the pre-trained PointGroup-PAPER model.

**Input**: `.las`, `.laz`, or `.ply` files in your input directory
**Output**: Segmented `.copc.laz` files in your output directory

Works in both Docker (JupyterLab) and local conda environments.

Reference: [Wielgosz et al. (2024)](https://doi.org/10.1016/j.rse.2024.114367)

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Inference requires a CUDA-capable GPU.")

In [ ]:
# Auto-detect paths (works in Docker and local installs)
import sys, os
from pathlib import Path

from sat.utils.paths import get_sat_root, get_input_dir, get_output_dir

SAT_ROOT = get_sat_root()
input_dir = get_input_dir()
output_dir = get_output_dir()

print(f"SAT_ROOT:   {SAT_ROOT}")
print(f"Input dir:  {input_dir}")
print(f"Output dir: {output_dir}")
print()

if not input_dir.exists():
    input_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created {input_dir} — copy your .las/.laz/.ply files here.")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Verify input files exist
files = list(input_dir.glob("*.las")) + list(input_dir.glob("*.laz")) + list(input_dir.glob("*.ply"))
print(f"Found {len(files)} input files:")
for f in files[:10]:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")
if len(files) > 10:
    print(f"  ... and {len(files) - 10} more")
if not files:
    print(f"No input files found! Copy your data to {input_dir}")

## Run Inference

This runs the full pipeline: coordinate transform → model inference → merge results → COPC conversion.

In [ ]:
import subprocess

result = subprocess.run(
    ["bash", "scripts/run_inference.sh", str(input_dir), str(output_dir), "true"],
    cwd=str(SAT_ROOT),
    capture_output=False,
    text=True
)
print(f"Exit code: {result.returncode}")

## Check Results

In [ ]:
results_dir = output_dir / "final_results"
results = sorted(results_dir.glob("*")) if results_dir.exists() else []
print(f"Output files ({len(results)}):")
for f in results:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")

if results:
    # Preview first result
    import laspy
    las = laspy.read(str(results[0]))
    print(f"\nPreview: {results[0].name}")
    print(f"  Points: {len(las.points):,}")
    print(f"  Dimensions: {list(las.point_format.dimension_names)}")
    if 'PredInstance' in las.point_format.dimension_names:
        import numpy as np
        n_trees = len(set(las.PredInstance)) - (1 if 0 in np.array(las.PredInstance) else 0)
        print(f"  Detected trees: {n_trees}")
else:
    print(f"No results found in {results_dir}")
    print("Make sure input files exist and inference completed successfully.")